# Self-Aligned Noise Augmentation (SEANA) in SSL Scientific Imaging

By Oankar Patil

TL;DR

- Traditionally, using noise as part of the augmentation process in SSL (Self-Supervised Learning), has been tried but shown to be not particularly useful or can even degrade learning representations in vision tasks.

- For example, if you go to the original SimCLR paper section 3 fig 5, you can see that pairing random cropping with Gaussian noise yielded noticeably lower ImageNet top-1 accuracy (50.3%) than pairing cropping with color distortion (56.3%) or Gaussian blur (53.4%). As such, the authors actually dropped using Gaussian noise.

- In this paper, we see that in scientific imaging regimes, there's typically acquistion noise that's intertwined with the semantic signal of the data. Furthermore, we don't necessarily even know the exact acquistion noise model / its parameters. Thus while you could denoise, you could also end up eroding semantic information OR not denoise enough and become reliant on the acquisiton noise itself....

- SEANA is thus an approach that allows us to denoise and inject *(acquistion-aligned)* independent noise into SSL views to aid the augmentation / view generation process without us having to know anything about the acquistion noise model within such scientific data!

- We found that across contrastive and non-contrastive SSL, SEANA's mechanisms of aligning itself with the underlying acquistion noise distribution and producing independent noise realizations, allows for significant improvements in learned semantic representations and in downstream top-1 linear probe accuracy as well as other benefits discussed in the main paper!


Finally:
- SEANA doesn't need dataset-specific denoiser training, doesn't need a denoiser or any additional module at inference, doesn't change SSL objectives so can be used with any joint-embedding SSL approach, and through caching its internal estimates, can take the exact same training time as baseline SSL.

The dataset used in this demo is BBBC048, a rea fluoresence imaging dataset of Jurkat Cells from the Broad Biomaging Institute.

BBBC048 has **no paired clean targets**. Denoised images are estimates, so visual smoothness is not proof of restored biological content. The quantitative comparison uses cell-phase classification with frozen backbones.

Links: [BBBC048](https://bbbc.broadinstitute.org/BBBC048) · [Colab resource FAQ](https://research.google.com/colaboratory/faq.html) · [experiment guide](fluorescence_imaging_bbbc048/README.md)

## 1. Load the data and helpers, then install SEANA from PyPI

Github Repo URL: https://github.com/opatil31/SEANA

In [ ]:
# @title Load code and the bundled BBBC048 dataset
REPO_URL = ""  # @param {type:"string"}
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import stat
import subprocess
import sys
import tempfile
import zipfile


def is_checkout(path):
    return (path / "fluorescence_imaging_bbbc048" / "demo.py").is_file() and (
        path / "requirements-colab.txt"
    ).is_file()


candidates = [Path.cwd(), Path("/content/SEANA"), Path("/content/SEANA_BBBC048")]
REPO_ROOT = next((p.resolve() for p in candidates if is_checkout(p)), None)
if REPO_ROOT is None:
    work = Path(tempfile.mkdtemp(prefix="seana-colab-"))
    if REPO_URL:
        if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+/?", REPO_URL):
            raise ValueError("Use a public https://github.com/OWNER/REPO URL, without credentials.")
        target = work / "repo"
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(target)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
        REPO_ROOT = target
    else:
        try:
            from google.colab import files
        except ImportError as error:
            raise RuntimeError(
                "Run from the repository root locally, or upload the bundle in Google Colab."
            ) from error
        print("Upload SEANA_BBBC048_Colab.zip (code + BBBC048 dataset).")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one repository/bundle ZIP.")
        archive_path = Path(next(iter(uploaded)))
        del uploaded
        with zipfile.ZipFile(archive_path) as archive:
            for item in archive.infolist():
                name = PurePosixPath(item.filename)
                if (
                    name.is_absolute()
                    or ".." in name.parts
                    or "\\" in item.filename
                    or stat.S_ISLNK(item.external_attr >> 16)
                ):
                    raise ValueError("Unsafe ZIP member: " + item.filename)
            archive.extractall(work)
        matches = [p.parent for p in work.rglob("requirements-colab.txt") if is_checkout(p.parent)]
        if len(matches) != 1:
            raise ValueError("Expected one SEANA checkout in the uploaded ZIP.")
        REPO_ROOT = matches[0]
    if not is_checkout(REPO_ROOT):
        raise ValueError("This repository does not contain the BBBC048 demo.")

bundle_manifest = REPO_ROOT / "colab_bundle.json"
if bundle_manifest.exists():
    for name, expected in json.loads(bundle_manifest.read_text())["files"].items():
        path = (REPO_ROOT / name).resolve()
        if REPO_ROOT.resolve() not in path.parents:
            raise ValueError("Invalid bundle manifest path.")
        raw = path.read_bytes()
        if len(raw) != expected["bytes"] or hashlib.sha256(raw).hexdigest() != expected["sha256"]:
            raise ValueError("Bundle checksum mismatch: " + name)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)

In [ ]:
# @title Install packages while retaining Colab's matching PyTorch/torchvision pair
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}
from importlib.metadata import distributions, version, PackageNotFoundError
import site

SEANA_VERSION = "0.1.3"

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is required.")
if INSTALL_DEPENDENCIES:
    pinned_runtime = []
    for package in ("torch", "torchvision", "numpy"):
        try:
            pinned_runtime.append(f"{package}=={version(package)}")
        except PackageNotFoundError as error:
            raise RuntimeError(
                "Use a Colab GPU runtime, or install the root requirements locally first."
            ) from error
    constraints = Path(tempfile.gettempdir()) / "seana-colab-constraints.txt"
    constraints.write_text("\n".join(pinned_runtime) + "\n")
    # Install the published wheel even if this runtime had an editable checkout.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall",
            "--no-deps", "--index-url", "https://pypi.org/simple", f"seana-ssl=={SEANA_VERSION}",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "-r",
            "requirements-colab.txt",
            "--constraint",
            str(constraints),
        ],
        check=True,
    )

# Import the installed wheel before any experiment helpers. A local seana.py
# otherwise takes precedence when this notebook is opened from a checkout.
package_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    package_dirs.insert(0, site.getusersitepackages())
installed = next(distributions(name="seana-ssl", path=package_dirs), None)
if installed is None:
    raise RuntimeError("Enable installation to install SEANA from PyPI.")
if installed.version != SEANA_VERSION:
    raise RuntimeError(f"Enable installation to use seana-ssl=={SEANA_VERSION}.")
for module in ("seana", "_seana_api"):
    expected = Path(installed.locate_file(module + ".py")).resolve()
    loaded = sys.modules.get(module)
    if not expected.is_file() or (loaded is not None and Path(loaded.__file__).resolve() != expected):
        raise RuntimeError("Restart the runtime and rerun setup to load SEANA from PyPI.")
sys.path.insert(0, str(Path(installed.locate_file("")).resolve()))
import seana
from seana import SEANA
if seana.__version__ != installed.version:
    raise RuntimeError("Restart the runtime after upgrading SEANA, then rerun setup.")
for module in ("seana", "_seana_api"):
    if Path(sys.modules[module].__file__).resolve() != Path(installed.locate_file(module + ".py")).resolve():
        raise RuntimeError("SEANA was loaded from the checkout. Restart the runtime and rerun setup.")
print(f"SEANA {installed.version} from PyPI: {seana.__file__}")

import torch
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from fluorescence_imaging_bbbc048 import demo
from fluorescence_imaging_bbbc048.config import PHASES
from fluorescence_imaging_bbbc048.dataset_archive import extract, verify

# Put downloaded model weights inside this checkout.
torch.hub.set_dir(str(REPO_ROOT / "weights" / "torch_hub"))
print({p: version(p) for p in ("seana-ssl", "torch", "torchvision", "lightly", "deepinv", "numpy")})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not connected")

# Enable the Colab widget frontend before progress bars and interactive controls.
try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()

In [ ]:
# @title Verify and extract the single-file BBBC048 dataset
archive = REPO_ROOT / "fluorescence_imaging_bbbc048/dataset/bbbc048_merged_v1.zip"
archive_sha = "3174f5672d721c98255ac16254c822a71efd0b934e4487bdab0702f2ec645ce1"
data_dir = REPO_ROOT / "data" / "bbbc048"
if not archive.is_file():
    raise FileNotFoundError("The upload must include the bundled BBBC048 dataset ZIP.")
if not data_dir.exists():
    manifest = extract(archive, data_dir, archive_sha)
else:
    verify(archive, archive_sha)
    manifest = data_dir / "manifest.json"
    if not manifest.is_file():
        raise RuntimeError(
            "The existing dataset extraction is incomplete; choose a fresh extraction directory."
        )
print("Dataset manifest:", manifest)

## 2. Choose a live experiment and inspect the training cells

Below, both presets default to **50 SSL epochs and 20 linear-probe epochs per arm**. The **long** preset uses all 25,813 training cells and 5,000 VST steps. Change the epoch controls below if you want a different training budget. Both presets use shared denoised estimates for speed and independent fresh noise after cropping. This explicit view order differs from the main `paper` recipe; the notebook remains a demo even with the full training set.

In [ ]:
# @title Experiment controls
PRESET = "quick"  # @param ["quick", "long"]
METHOD = "simclr"  # @param ["simclr", "vicreg", "byol", "mocov2", "lejepa"]
SSL_EPOCHS = 50  # @param {type:"integer"}
PROBE_EPOCHS = 20  # @param {type:"integer"}
SIGMA_VST = 0.8  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}
RUN_COMPARISON = True  # @param {type:"boolean"}
ALLOW_SLOW_CPU = False  # @param {type:"boolean"}

from datetime import datetime, timezone
from uuid import uuid4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not ALLOW_SLOW_CPU:
    raise RuntimeError(
        "Select Runtime → Change runtime type → GPU, reconnect, and rerun. CPU training is slow."
    )
cfg, train_limit = demo.recipe(
    manifest, preset=PRESET, method=METHOD, seed=SEED, sigma=SIGMA_VST, device=str(DEVICE),
    ssl_epochs=SSL_EPOCHS, probe_epochs=PROBE_EPOCHS,
)
train, test, dataset = demo.prepare_data(cfg, train_limit)
OUTPUT = (
    REPO_ROOT
    / "runs"
    / "colab"
    / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8])
)
print("Training cells:", len(train), "| Test cells:", len(test))
print("SSL epochs:", cfg.epochs, "| Probe epochs:", cfg.probe_epochs, "| VST steps:", cfg.seana_fit_steps)
print("This experiment's resampling strength:", cfg.sigma_vst)
print("Output:", OUTPUT)
print("Phase             Train    Test")
for phase in PHASES:
    print(
        f"{phase:15s} {dataset['class_counts']['train'][phase]:6d} {dataset['class_counts']['test'][phase]:7d}"
    )

In [ ]:
# One deterministically chosen TRAINING cell per phase, before any fitting.
example_indices = [
    next(j for j, i in enumerate(train.indices) if train.dataset.records[i]["label"] == k)
    for k in range(len(PHASES))
]
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5), constrained_layout=True)
for ax, phase, index in zip(axes, PHASES, example_indices):
    image, _ = train[index]
    ax.imshow(image.permute(1, 2, 0).numpy(), vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(phase)
    ax.axis("off")
plt.show()

## 4. Fit the VST without clean targets

The helper constructs the PyPI package's `SEANA` module and calls its `.fit()` method on training cells. This step downloads DeepInverse's pretrained **DRUNet** on first use. DRUNet stays frozen. Only the small VST is fitted, using acquired training images and the same self-supervised fitting objective as the release. Neither test images nor phase labels enter that objective. The fitted module is available as `augmenter` below, and `treatment.pt` can be restored with `SEANA.load(path)` for use in your own pipeline.

In [ ]:
from tqdm.auto import tqdm

with tqdm(total=100, desc="Fitting VST on training cells", unit="%") as progress:
    treatment = demo.fit_seana(
        cfg, train, dataset, OUTPUT, DEVICE, progress=lambda percent: progress.update(percent - progress.n)
    )
augmenter: SEANA = treatment.augmenter  # A fitted seana.SEANA instance from PyPI.
print("VST fitted. DRUNet and VST are now frozen.")

## 5. Play with the noise-aligned views

The slider changes **only the preview**. Geometry stays fixed so you can isolate the effect of resampling. Both views share a denoised estimate but receive independent noise in VST space. At strength zero, they equal the estimate. All panels use the same [0,1] intensity scale.

The displayed cells come from training. The comparison in the next section uses the fixed `SIGMA_VST` from section 2, not the last slider position. To run a different experiment, change section 2 and rerun from there; avoid choosing a setting based on repeated test scores.

In [ ]:
import ipywidgets as widgets


def show_views(phase=PHASES[0], sigma=SIGMA_VST, noise_seed=0):
    image, _ = train[example_indices[PHASES.index(phase)]]
    views = demo.preview(treatment, image, sigma, noise_seed)
    fig, axes = plt.subplots(1, 5, figsize=(14, 3), constrained_layout=True)
    titles = ["Acquired", "DRUNet alone", "Learned VST + DRUNet", "Resampled view A", "Resampled view B"]
    for ax, (name, tensor), title in zip(axes, views.items(), titles):
        ax.imshow(tensor.permute(1, 2, 0).numpy(), vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"Training cell: {phase} | preview sigma_vst={sigma:.2f} | noise seed={noise_seed}")
    plt.show()


# Render once even in environments that cannot display widgets.
show_views()
controls = dict(
    phase=widgets.Dropdown(options=list(PHASES), description="Cell phase"),
    sigma=widgets.FloatSlider(
        value=SIGMA_VST, min=0, max=2, step=0.05, description="sigma_vst", continuous_update=False
    ),
    noise_seed=widgets.IntSlider(value=0, min=0, max=20, description="Noise seed", continuous_update=False),
)
interactive = widgets.interactive_output(show_views, controls)
display(widgets.VBox(list(controls.values())), interactive)

In [ ]:
# Inspect the learned transformation in normalized intensity coordinates.
with torch.inference_mode():
    x = torch.linspace(0, 1, 256, device=DEVICE)
    y = treatment.vst.spline1(x.reshape(1, 1, 16, 16)).flatten().cpu()
fig, ax = plt.subplots(figsize=(5, 3.5), constrained_layout=True)
ax.plot(x.cpu(), y, label="Fitted VST", color="#0f766e", linewidth=2)
ax.plot([0, 1], [0, 1], "--", color="#64748b", label="Identity reference")
ax.set(xlabel="Normalized input intensity", ylabel="VST-space intensity", title="The learned transform")
ax.legend()
plt.show()

## 6. Measure what the representation learned

Two randomly initialized ResNet-18 backbones are trained from the **same starting weights**. Baseline uses ordinary spatial views. SEANA uses its fitted estimate and independently resampled views. SSL losses are different training signals, so lower loss is not itself evidence of better representations.

For each frozen backbone, acquired-image features are extracted once and cached for a linear classifier. This avoids rerunning the backbone during every probe epoch and keeps probe inputs identical across treatments.

Uncheck `RUN_COMPARISON` in section 2 if you only want the visual demo.

In [ ]:
if RUN_COMPARISON:
    result = demo.compare(cfg, train, test, dataset, treatment, OUTPUT, DEVICE)
    fig = demo.plot_results(result)
    fig.savefig(OUTPUT / "comparison.png", dpi=160)
    plt.show()
    print("SEANA minus baseline, percentage points:")
    for metric, delta in result["delta_percentage_points"].items():
        print(f"  {metric:20s} {delta:+.2f}")
    print("These are this run's observed scores, not promised improvements or paper results.")
else:
    result = None
    print("Comparison skipped; the fitted VST and visual playground remain available.")